In [0]:
from pyspark.sql.functions import *

# READ TABLES
trip_df = spark.table("default.trips")
station_df = spark.table("default.stations")
payment_df = spark.table("default.payments")

# RENAME COLUMNS

trip_df = trip_df.toDF(
    "trip_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "rider_id"
)

station_df = station_df.toDF(
    "station_id",
    "station_name",
    "latitude",
    "longitude"
)

payment_df = payment_df.toDF(
    "payment_id",
    "payment_date",
    "amount",
    "rider_id"
)

# SHOW DATA

trip_df.show(5)
station_df.show(5)
payment_df.show(5)

# DATE DIMENSION

dim_date = trip_df.withColumn(
    "date",
    to_date(col("started_at"))
).select("date").distinct() \
.withColumn("month", month("date")) \
.withColumn("year", year("date")) \
.withColumn("quarter", quarter("date"))

# FACT TRIP

fact_trip = trip_df.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp(col("ended_at")) -
        unix_timestamp(col("started_at"))
    ) / 60
)

# FACT PAYMENT

fact_payment = payment_df

# SAVE TABLES

station_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("dim_station")

dim_date.write.format("delta") \
.mode("overwrite") \
.saveAsTable("dim_date")

fact_trip.write.format("delta") \
.mode("overwrite") \
.saveAsTable("fact_trip")

fact_payment.write.format("delta") \
.mode("overwrite") \
.saveAsTable("fact_payment")

# SHOW TABLES

spark.sql("SHOW TABLES").show()

# ANALYSIS

fact_trip.groupBy().avg("trip_duration_minutes").show()

fact_payment.show(5)
spark.sql("SHOW TABLES").show()

fact_trip.groupBy().avg("trip_duration_minutes").show()

+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|         trip_id|rideable_type|         started_at|           ended_at|start_station_id|end_station_id|rider_id|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|05961376528D5AA9| classic_bike|2021-06-22 11:35:53|2021-06-22 11:44:04|    KA1504000146|  KA1504000148|   50304|
|042029FFA981571B| classic_bike|2021-06-10 16:10:33|2021-06-10 16:26:43|           13192|         13071|   70734|
|F0E9906B452FC22A|  docked_bike|2021-06-05 22:44:09|2021-06-05 23:04:56|    TA1306000015|  KA1504000135|   71975|
|D5548D6B05F9DB39| classic_bike|2021-06-18 18:24:07|2021-06-18 18:54:44|    TA1308000012|  TA1307000128|   27022|
|0CDF8C5894FDCAAB| classic_bike|2021-06-17 19:33:49|2021-06-17 19:42:28|           15530|  KA1504000135|   38406|
+----------------+-------------+-------------------+-------------------+----------------

In [0]:
# READ RIDERS TABLE
rider_df = spark.table("default.riders")

# RENAME COLUMNS
rider_df = rider_df.toDF(
    "rider_id",
    "first_name",
    "last_name",
    "address",
    "birthday",
    "account_start_date",
    "is_member"
)

# SHOW DATA
rider_df.show(5)

# SAVE TABLE
rider_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("dim_rider")

# SHOW TABLES
spark.sql("SHOW TABLES").show()

+--------+----------+---------+--------------------+----------+------------------+---------+
|rider_id|first_name|last_name|             address|  birthday|account_start_date|is_member|
+--------+----------+---------+--------------------+----------+------------------+---------+
|    1001|  Jennifer|    Smith|     397 Diana Ferry|1976-08-10|        2019-11-01|     true|
|    1002|     Karen|    Smith|644 Brittany Row ...|1998-08-10|        2022-02-04|     true|
|    1003|     Bryan|  Roberts|996 Dickerson Tur...|1999-03-29|        2019-08-26|    false|
|    1004|     Jesse|Middleton|7009 Nathan Expre...|1969-04-11|        2019-09-14|     true|
|    1005| Christine|Rodriguez|224 Washington Mi...|1974-08-27|        2020-03-24|    false|
+--------+----------+---------+--------------------+----------+------------------+---------+
only showing top 5 rows
+--------+------------+-----------+
|database|   tableName|isTemporary|
+--------+------------+-----------+
| default|    dim_date|      fa

In [0]:
from pyspark.sql.functions import *

# READ RIDER TABLE

rider_df = spark.table("default.riders").toDF(
    "rider_id",
    "first_name",
    "last_name",
    "address",
    "birthday",
    "account_start_date",
    "is_member"
)

# ADD RIDER AGE TO FACT TRIP

fact_trip = fact_trip.join(
    rider_df.select("rider_id", "birthday"),
    on="rider_id",
    how="left"
)

fact_trip = fact_trip.withColumn(
    "rider_age",
    year(col("started_at")) - year(col("birthday"))
)

# SHOW OUTPUT

fact_trip.select(
    "trip_id",
    "rider_id",
    "started_at",
    "rider_age",
    "trip_duration_minutes"
).show(5)

+----------------+--------+-------------------+---------+---------------------+
|         trip_id|rider_id|         started_at|rider_age|trip_duration_minutes|
+----------------+--------+-------------------+---------+---------------------+
|05961376528D5AA9|   50304|2021-06-22 11:35:53|       22|    8.183333333333334|
|042029FFA981571B|   70734|2021-06-10 16:10:33|       26|   16.166666666666668|
|F0E9906B452FC22A|   71975|2021-06-05 22:44:09|       42|   20.783333333333335|
|D5548D6B05F9DB39|   27022|2021-06-18 18:24:07|       37|   30.616666666666667|
|0CDF8C5894FDCAAB|   38406|2021-06-17 19:33:49|       35|                 8.65|
+----------------+--------+-------------------+---------+---------------------+
only showing top 5 rows


In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------+-----------+
|database|   tableName|isTemporary|
+--------+------------+-----------+
| default|    dim_date|      false|
| default|   dim_rider|      false|
| default| dim_station|      false|
| default|fact_payment|      false|
| default|   fact_trip|      false|
| default|    payments|      false|
| default|      riders|      false|
| default|    stations|      false|
| default|       trips|      false|
+--------+------------+-----------+

